# **Customer Support Intent Classification by Chaudhary Hadi**

In [1]:
# Libraries Install aur Import
!pip install transformers datasets torch accelerate

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import torch



## **LOADING BANKING77 DATASET**

In [2]:
# Banking77 Dataset Load aur Basic Info
print("="*50)
print("LOADING BANKING77 DATASET")
print("="*50)

# Dataset load
dataset = load_dataset("banking77")

# Basic info
print(f"\n Dataset Structure:")
print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

# Pehle 2 examples dekhte hain
print(f"\n Sample Training Examples:")
for i in range(2):
    print(f"\nExample {i+1}:")
    print(f"Text: {dataset['train'][i]['text']}")
    print(f"Label: {dataset['train'][i]['label']}")

# Total classes
num_labels = len(set(dataset['train']['label']))
print(f"\n Total Classes: {num_labels}")

# Label names ka dictionary
label_names = dataset['train'].features['label'].names
print(f"\n First 5 Label Names:")
for i in range(5):
    print(f"  {i}: {label_names[i]}")

LOADING BANKING77 DATASET


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]


📊 Dataset Structure:
Training samples: 10003
Test samples: 3080

📝 Sample Training Examples:

Example 1:
Text: I am still waiting on my card?
Label: 11

Example 2:
Text: What can I do if my card still hasn't arrived after 2 weeks?
Label: 11

🏷️ Total Classes: 77

📋 First 5 Label Names:
  0: activate_my_card
  1: age_limit
  2: apple_pay_or_google_pay
  3: atm_support
  4: automatic_top_up


## **LOADING TOKENIZER AND MODEL**

In [3]:
  Tokenizer aur Model Load
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("="*50)
print("LOADING TOKENIZER AND MODEL")
print("="*50)

# Tokenizer load
model_name = "bert-base-uncased"
print(f"\n Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Model load with 77 classes
print(f"\n Loading model: {model_name}")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=77  # Banking77 has 77 classes
)

# Check tokenizer
print(f"\n Tokenizer loaded successfully!")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

# Test tokenizer on sample text
sample_text = "I lost my credit card"
tokens = tokenizer(sample_text)
print(f"\n Tokenizer Test:")
print(f"Original text: '{sample_text}'")
print(f"Tokenized: {tokens['input_ids']}")
print(f"Decoded back: {tokenizer.decode(tokens['input_ids'])}")

LOADING TOKENIZER AND MODEL

📥 Loading tokenizer: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


📥 Loading model: bert-base-uncased


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Tokenizer loaded successfully!
Vocabulary size: 30522
Max length: 512

🔤 Tokenizer Test:
Original text: 'I lost my credit card'
Tokenized: [101, 1045, 2439, 2026, 4923, 4003, 102]
Decoded back: [CLS] i lost my credit card [SEP]


## **TOKENIZING DATASET**

In [4]:
# Data Preprocessing - Tokenization Function
print("="*50)
print("TOKENIZING DATASET")
print("="*50)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",    # All sequences same length
        truncation=True,          # Cut long sequences
        max_length=128            # Banking queries are short
    )

# Apply tokenization to dataset
print(" Tokenizing training data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

print(" Tokenization complete!")
print(f"\n Tokenized Dataset Structure:")
print(f"Features: {tokenized_datasets['train'].column_names}")
print(f"Sample input_ids shape: {len(tokenized_datasets['train'][0]['input_ids'])} tokens")

# Show sample
print(f"\n Sample after tokenization:")
print(f"Text: {tokenized_datasets['train'][0]['text'][:50]}...")
print(f"Labels: {tokenized_datasets['train'][0]['label']}")
print(f"input_ids (first 10): {tokenized_datasets['train'][0]['input_ids'][:10]}")
print(f"attention_mask (first 10): {tokenized_datasets['train'][0]['attention_mask'][:10]}")

TOKENIZING DATASET
📥 Tokenizing training data...


Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

✅ Tokenization complete!

📊 Tokenized Dataset Structure:
Features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Sample input_ids shape: 128 tokens

🔍 Sample after tokenization:
Text: I am still waiting on my card?...
Labels: 11
input_ids (first 10): [101, 1045, 2572, 2145, 3403, 2006, 2026, 4003, 1029, 102]
attention_mask (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## **GPU Checking...**

In [5]:
print("Libraries installed aur imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries installed aur imported successfully!
PyTorch version: 2.10.0+cu128
CUDA available: True


## **SETTING UP TRAINING ARGUMENTS**

In [7]:
# Training Arguments Configuration
print("="*50)
print("SETTING UP TRAINING ARGUMENTS")
print("="*50)

# Training arguments
training_args = TrainingArguments(
    output_dir="./banking77-bert-results",     # Model save location
    num_train_epochs=3,                         # Total training rounds
    per_device_train_batch_size=16,              # Batch size per GPU
    per_device_eval_batch_size=16,               # Evaluation batch size
    warmup_steps=500,                            # Learning rate warmup
    weight_decay=0.01,                           # Weight decay for regularization
    logging_dir="./logs",                         # Logs directory
    logging_steps=100,                            # Log every 100 steps
    eval_strategy="epoch",                         #Fixed: evaluation_strategy → eval_strategy
    save_strategy="epoch",                         # Save after each epoch
    load_best_model_at_end=True,                   # Load best model at end
    metric_for_best_model="accuracy",               # Use accuracy for best model
    report_to="none",                               # Disable wandb/tensorboard
)

print(" Training arguments configured!")
print(f"\n Training Configuration:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")  # Default: 5e-5
print(f"Warmup steps: {training_args.warmup_steps}")
print(f"Evaluation strategy: {training_args.eval_strategy}")  #  Fixed variable name
print(f"Save strategy: {training_args.save_strategy}")

SETTING UP TRAINING ARGUMENTS


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training arguments configured!

📋 Training Configuration:
Epochs: 3
Batch size: 16
Learning rate: 5e-05
Warmup steps: 500
Evaluation strategy: IntervalStrategy.EPOCH
Save strategy: SaveStrategy.EPOCH


## **SETTING UP TRAINER AND METRICS**

In [9]:
# Setup Trainer and Evaluation Metrics
print("="*50)
print("SETTING UP TRAINER AND METRICS")
print("="*50)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Define compute metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print(" Metrics function defined!")

# Initialize Trainer - WITHOUT tokenizer parameter
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,  # Added metrics
)

print("\n Trainer initialized successfully!")
print(f"\n Training Dataset Size: {len(tokenized_datasets['train'])} samples")
print(f" Evaluation Dataset Size: {len(tokenized_datasets['test'])} samples")
print("\n Ready to start training!")
print("\n Next step: Run Cell 7 to start training!")

SETTING UP TRAINER AND METRICS
✅ Metrics function defined!

✅ Trainer initialized successfully!

📊 Training Dataset Size: 10003 samples
📊 Evaluation Dataset Size: 3080 samples

🎯 Ready to start training!

➡️ Next step: Run Cell 7 to start training!


## **STARTING TRAINING...**

In [10]:
# Cell 7: TRAINING - Yahan actual fine-tuning hota hai
print("="*50)
print(" STARTING TRAINING...")
print("="*50)
print(" This will take 5-10 minutes on T4 GPU")
print("="*50)

# Start training
trainer.train()

print("="*50)
print(" TRAINING COMPLETE!")
print("="*50)

# Save the model
trainer.save_model("./banking77-bert-finetuned")
tokenizer.save_pretrained("./banking77-bert-finetuned")

print("\n Model saved to: ./banking77-bert-finetuned")

🚀 STARTING TRAINING...
⚠️ This will take 5-10 minutes on T4 GPU


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.566600,1.190378,0.785065,0.785093,0.785065,0.761313
2,0.399943,0.405925,0.913961,0.919603,0.913961,0.914136
3,0.193817,0.307662,0.929870,0.932991,0.929870,0.929915


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

✅ TRAINING COMPLETE!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Model saved to: ./banking77-bert-finetuned


## **TESTING FINE-TUNED MODEL**

In [12]:
# Cell 8: TESTING - Apni queries check karo
print("="*50)
print("TESTING FINE-TUNED MODEL")
print("="*50)

# Get label names
label_names = dataset['train'].features['label'].names

# Test queries
test_queries = [
    "I lost my credit card, what should I do?",
    "Can you increase my credit limit?",
    "What's the interest rate for savings account?",
    "My ATM card is not working",
    "How do I activate my new card?"
]

print("\n Testing custom queries:\n")
for query in test_queries:
    # Tokenize
    inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=128)

    # Move to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()

    print(f" Query: '{query}'")
    print(f" Intent: {label_names[predicted_class]}")
    print(f" Confidence: {confidence:.2%}")
    print("-" * 50)

🧪 TESTING FINE-TUNED MODEL

📝 Testing custom queries:

❓ Query: 'I lost my credit card, what should I do?'
🎯 Intent: lost_or_stolen_card
📊 Confidence: 82.01%
--------------------------------------------------
❓ Query: 'Can you increase my credit limit?'
🎯 Intent: automatic_top_up
📊 Confidence: 58.71%
--------------------------------------------------
❓ Query: 'What's the interest rate for savings account?'
🎯 Intent: exchange_rate
📊 Confidence: 82.87%
--------------------------------------------------
❓ Query: 'My ATM card is not working'
🎯 Intent: declined_cash_withdrawal
📊 Confidence: 86.22%
--------------------------------------------------
❓ Query: 'How do I activate my new card?'
🎯 Intent: activate_my_card
📊 Confidence: 97.61%
--------------------------------------------------


## **🔧 IMPROVED TESTING WITH THRESHOLD**

In [14]:
# IMPROVE PREDICTIONS - Better testing with confidence threshold
print("="*50)
print("IMPROVED TESTING WITH THRESHOLD")
print("="*50)

def predict_intent(query, threshold=0.7):
    # Tokenize
    inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=128)

    # Move to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        max_prob, predicted_class = torch.max(probabilities, dim=-1)
        confidence = max_prob.item()
        predicted_idx = predicted_class.item()

    # Check confidence threshold
    if confidence < threshold:
        return "LOW CONFIDENCE - Needs human review", confidence, predicted_idx
    else:
        return label_names[predicted_idx], confidence, predicted_idx

# Test again
test_queries = [
    "I lost my credit card, what should I do?",
    "Can you increase my credit limit?",
    "What's the interest rate for savings account?",
    "My ATM card is not working",
    "How do I activate my new card?",
    "I forgot my PIN"  # Extra test
]

print(f"\n Testing with 70% confidence threshold:\n")  # Fixed: Removed {} around 70%

for query in test_queries:
    intent, confidence, idx = predict_intent(query)

    print(f"Query: '{query}'")
    print(f"Intent: {intent}")
    print(f"Confidence: {confidence:.2%}")

    # Show top 3 predictions for low confidence
    if confidence < 0.7:
        print("   Top 3 alternatives:")
        inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=128)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            top3 = torch.topk(probs, 3)
            for i in range(3):
                print(f"   {i+1}. {label_names[top3.indices[i].item()]}: {top3.values[i].item():.2%}")
    print("-" * 50)

🔧 IMPROVED TESTING WITH THRESHOLD

📝 Testing with 70% confidence threshold:

❓ Query: 'I lost my credit card, what should I do?'
🎯 Intent: lost_or_stolen_card
📊 Confidence: 82.01%
--------------------------------------------------
❓ Query: 'Can you increase my credit limit?'
🎯 Intent: 🔴 LOW CONFIDENCE - Needs human review
📊 Confidence: 58.71%
   Top 3 alternatives:
   1. automatic_top_up: 58.71%
   2. top_up_limits: 13.24%
   3. topping_up_by_card: 5.74%
--------------------------------------------------
❓ Query: 'What's the interest rate for savings account?'
🎯 Intent: exchange_rate
📊 Confidence: 82.87%
--------------------------------------------------
❓ Query: 'My ATM card is not working'
🎯 Intent: declined_cash_withdrawal
📊 Confidence: 86.22%
--------------------------------------------------
❓ Query: 'How do I activate my new card?'
🎯 Intent: activate_my_card
📊 Confidence: 97.61%
--------------------------------------------------
❓ Query: 'I forgot my PIN'
🎯 Intent: pin_blocked
📊 

## **☁️ SAVING MODEL TO HUGGING FACE HUB**

In [16]:
# Save to Hugging Face Hub
print("="*50)
print(" SAVING MODEL TO HUGGING FACE HUB")
print("="*50)

# Login to Hugging Face (first time only)
from huggingface_hub import notebook_login

print("\n Please login to Hugging Face:")
print("1. Go to: https://huggingface.co/settings/tokens")
print("2. Create new token (write permission)")
print("3. Paste token below\n")

notebook_login()

# Save model to hub
model_name = "banking77-bert-finetuned"  # Change this to your username/model-name
model.push_to_hub(model_name)
tokenizer.push_to_hub(model_name)

print(f"\n Model uploaded to: https://huggingface.co/{model_name}")

☁️ SAVING MODEL TO HUGGING FACE HUB

🔑 Please login to Hugging Face:
1. Go to: https://huggingface.co/settings/tokens
2. Create new token (write permission)
3. Paste token below



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mgesbxh/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md: 0.00B [00:00, ?B/s]


✅ Model uploaded to: https://huggingface.co/banking77-bert-finetuned


## **🎯 READY-TO-USE CLASSIFICATION FUNCTION**

In [17]:
# READY-TO-USE FUNCTION for presentations
print("="*50)
print(" READY-TO-USE CLASSIFICATION FUNCTION")
print("="*50)

def classify_banking_query(query, show_details=True):
    """
    Simple function to classify any banking query
    """
    # Predict
    inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=128)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
        confidence, pred = torch.max(probs, dim=-1)
        confidence = confidence.item()
        pred_class = pred.item()

    intent = label_names[pred_class]

    # Show results
    if show_details:
        print(f"\n Query: {query}")
        print(f" Intent: {intent}")
        print(f" Confidence: {confidence:.1%}")

        if confidence < 0.7:
            print(" Low confidence - escalate to human")

        # Show top 3
        top3 = torch.topk(probs, 3)
        print("\nTop 3 predictions:")
        for i in range(3):
            print(f"   {i+1}. {label_names[top3.indices[i].item()]}: {top3.values[i].item():.1%}")

    return intent, confidence

# Demo
print("\n Try it yourself!")
print("-" * 50)

while True:
    user_query = input("\nEnter your banking query (or 'quit' to exit): ")
    if user_query.lower() == 'quit':
        break
    classify_banking_query(user_query)
    print("-" * 50)

🎯 READY-TO-USE CLASSIFICATION FUNCTION

🔍 Try it yourself!
--------------------------------------------------

Enter your banking query (or 'quit' to exit): I forget my 4 digit pin

📝 Query: I forget my 4 digit pin
🎯 Intent: pin_blocked
📊 Confidence: 77.2%

Top 3 predictions:
   1. pin_blocked: 77.2%
   2. get_physical_card: 17.1%
   3. change_pin: 0.8%
--------------------------------------------------

Enter your banking query (or 'quit' to exit): quit
